In [21]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

### Label construction and candidate generation for movie reranker training pairs

In [24]:
DATA_ROOT = Path("../../data")
ARTIFACT_DIR = Path("../../ml/artifacts/content_recommendation_v2")

INTERACTIONS_PATH = DATA_ROOT / "processed" / "interactions_clean.parquet"
ITEM_FEATURES_PARQUET = DATA_ROOT / "processed" / "item_features.parquet"
ITEM_FEATURES_CSV = DATA_ROOT / "processed" / "item_features.csv"
METADATA_CSV = ARTIFACT_DIR / "movie_metadata.csv"
EMBEDDINGS_PATH = ARTIFACT_DIR / "overview_embeddings.npy"
OUTPUT_PATH = DATA_ROOT / "processed" / "reranker_pairs.parquet"

RNG = np.random.default_rng(42)

# MovieLens ratings are 0.5–5.0; positive threshold ≥ 4.0
POSITIVE_THRESHOLD = 4.0

# --- Stage 1 simulation config ---
STAGE1_CANDIDATES_PER_QUERY = 150   # top-K from Stage 1 multi-feature score
STAGE1_RETRIEVAL_POOL = 300         # initial embedding retrieval pool before scoring
STAGE1_BATCH_SIZE = 256             # queries processed in one matmul batch

# --- Graded relevance mapping (MovieLens 0.5-5.0 scale) ---
# LambdaRank natively supports graded labels for finer ranking signal
def rating_to_relevance(rating: float) -> int:
    """Convert MovieLens rating to graded relevance for LambdaRank."""
    if rating >= 5.0:
        return 4
    elif rating >= 4.5:
        return 3
    elif rating >= 4.0:
        return 2
    elif rating >= 3.0:
        return 1
    else:
        return 0

# Memory guards
MAX_POSITIVES_PER_USER = 30
MAX_USERS_FOR_TRAINING = 30000
MAX_TOTAL_PAIRS = 15_000_000        # raised — Stage 1 lists are dense but uniform size
MIN_POSITIVES_PER_GROUP = 1         # require ≥1 relevant item (relevance ≥ 2)
MIN_NEGATIVES_PER_GROUP = 3         # require ≥3 non-relevant items

In [25]:
# ------------------------------------------------------------
# 1) Load interactions and construct labels
interactions = pd.read_parquet(INTERACTIONS_PATH, columns=["user_id", "movie_id", "rating"]).copy()
interactions["user_id"] = pd.to_numeric(interactions["user_id"], errors="coerce")
interactions["movie_id"] = pd.to_numeric(interactions["movie_id"], errors="coerce")
interactions["rating"] = pd.to_numeric(interactions["rating"], errors="coerce")
interactions = interactions.dropna(subset=["user_id", "movie_id", "rating"]).copy()
interactions["user_id"] = interactions["user_id"].astype("int64")
interactions["movie_id"] = interactions["movie_id"].astype("int64")
interactions["rating"] = interactions["rating"].astype("float32")

# MovieLens ratings are 0.5–5.0 (half-star scale); no zero-rated rows to drop
interactions = interactions[(interactions["rating"] >= 0.5) & (interactions["rating"] <= 5.0)].copy()

# Binary label: >=4.0 → 1 (positive), <4.0 → 0 (negative)
interactions["label"] = (interactions["rating"] >= POSITIVE_THRESHOLD).astype("int8")

# Stratified user sampling: sample from activity-level bins for diversity
user_activity = interactions["user_id"].value_counts()
available_users = len(user_activity)

if available_users > MAX_USERS_FOR_TRAINING:
    N_STRATA = min(10, int(user_activity.nunique()))
    if N_STRATA > 1:
        activity_bins = pd.qcut(user_activity, q=N_STRATA, labels=False, duplicates="drop")
        unique_bins = sorted(activity_bins.dropna().unique())
        selected_users = []
        remaining_slots = MAX_USERS_FOR_TRAINING

        for i, bin_id in enumerate(unique_bins):
            bin_members = user_activity.index[activity_bins == bin_id].to_numpy()
            bins_left = len(unique_bins) - i
            take = min(len(bin_members), max(1, remaining_slots // bins_left))
            chosen = RNG.choice(bin_members, size=take, replace=False)
            selected_users.extend(chosen.tolist())
            remaining_slots -= take

        if len(selected_users) < MAX_USERS_FOR_TRAINING:
            already = set(selected_users)
            extra = [u for u in user_activity.index if u not in already][
                : MAX_USERS_FOR_TRAINING - len(selected_users)
            ]
            selected_users.extend(extra)

        selected_users = set(selected_users)
    else:
        selected_users = set(user_activity.index[:MAX_USERS_FOR_TRAINING].tolist())

    interactions = interactions[interactions["user_id"].isin(selected_users)].copy()
    del selected_users

del user_activity
interactions = interactions.reset_index(drop=True)

positives = interactions[interactions["label"] == 1].copy()
negatives = interactions[interactions["label"] == 0].copy()

print(f"Users kept for training pair generation: {interactions['user_id'].nunique():,}")
print(f"Interactions after filtering: {len(interactions):,}")
print("Label distribution:")
print(interactions["label"].value_counts().sort_index())

Users kept for training pair generation: 30,000
Interactions after filtering: 2,833,361
Label distribution:
label
0    1413417
1    1419944
Name: count, dtype: int64


In [26]:
# ------------------------------------------------------------
# 2) Load item features for popularity negatives
if ITEM_FEATURES_CSV.exists():
    item_features = pd.read_csv(ITEM_FEATURES_CSV)
elif ITEM_FEATURES_PARQUET.exists():
    item_features = pd.read_parquet(ITEM_FEATURES_PARQUET)
else:
    raise FileNotFoundError("Missing item_features file (csv or parquet)")

item_features["movie_id"] = pd.to_numeric(item_features["movie_id"], errors="coerce")
item_features = item_features.dropna(subset=["movie_id"]).copy()
item_features["movie_id"] = item_features["movie_id"].astype("int64")

if "popularity_rank" not in item_features.columns:
    raise ValueError("item_features must include popularity_rank")

item_features["popularity_rank"] = pd.to_numeric(item_features["popularity_rank"], errors="coerce")
item_features = item_features.dropna(subset=["popularity_rank"]).copy()

top_popular_ids = (
    item_features.sort_values("popularity_rank", ascending=True)["movie_id"]
    .drop_duplicates()
    .head(500)
    .tolist()
)

In [27]:
# ------------------------------------------------------------
# 3) Load embeddings + metadata, build Stage 1 candidate cache
import re, time

if not METADATA_CSV.exists():
    raise FileNotFoundError(f"Missing movie metadata: {METADATA_CSV}")

metadata = pd.read_csv(METADATA_CSV)
if "id" not in metadata.columns or "movie_index" not in metadata.columns:
    raise ValueError("Metadata must contain id and movie_index")

metadata["id"] = pd.to_numeric(metadata["id"], errors="coerce")
metadata["movie_index"] = pd.to_numeric(metadata["movie_index"], errors="coerce")
metadata = metadata.dropna(subset=["id", "movie_index"]).copy()
metadata["id"] = metadata["id"].astype("int64")
metadata["movie_index"] = metadata["movie_index"].astype("int64")

embeddings = np.load(EMBEDDINGS_PATH)
if embeddings.dtype == np.float16:
    embeddings = embeddings.astype(np.float32)
if embeddings.ndim != 2:
    raise ValueError(f"Expected 2D embeddings, got shape={embeddings.shape}")
embeddings_f32 = embeddings if embeddings.dtype == np.float32 else embeddings.astype(np.float32)

movie_to_index = dict(zip(metadata["id"].tolist(), metadata["movie_index"].tolist()))
index_to_movie = dict(zip(metadata["movie_index"].tolist(), metadata["id"].tolist()))

# --- Parse metadata for Stage 1 scoring ---
def _parse_set_field(value):
    if pd.isna(value):
        return set()
    text = str(value).strip()
    if not text:
        return set()
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = json.loads(text.replace("'", '"'))
            if isinstance(parsed, list):
                return {str(x).strip().lower() for x in parsed if str(x).strip()}
        except Exception:
            pass
    return {part.strip().lower() for part in text.split(",") if part.strip()}

def _jaccard(a, b):
    if not a and not b:
        return 0.0
    union = a | b
    return len(a & b) / len(union) if union else 0.0

def _actor_overlap(a_set, b_set):
    if not a_set or not b_set:
        return 0.0
    a_list, b_list = list(a_set), list(b_set)
    score = 0.0
    for i, actor in enumerate(a_list):
        if actor in b_set:
            j = b_list.index(actor) if actor in b_list else len(b_list)
            score += (1.0 / (i + 1)) * (1.0 / (j + 1))
    max_possible = sum(1.0 / (i + 1) ** 2 for i in range(min(len(a_list), len(b_list))))
    return score / max_possible if max_possible > 0 else 0.0

# Pre-parse catalog metadata into arrays indexed by embedding index
catalog_md = pd.read_csv(
    METADATA_CSV,
    usecols=["id", "movie_index", "genres", "keywords", "top_5_actors",
             "director", "collection_name"],
)
catalog_md["id"] = pd.to_numeric(catalog_md["id"], errors="coerce")
catalog_md["movie_index"] = pd.to_numeric(catalog_md["movie_index"], errors="coerce")
catalog_md = catalog_md.dropna(subset=["id", "movie_index"]).copy()
catalog_md["id"] = catalog_md["id"].astype("int64")
catalog_md["movie_index"] = catalog_md["movie_index"].astype("int64")

n_movies = embeddings_f32.shape[0]
genre_sets_arr = [set()] * n_movies
keyword_sets_arr = [set()] * n_movies
actor_sets_arr = [set()] * n_movies
director_arr = [""] * n_movies
collection_arr = [""] * n_movies

for _, row in catalog_md.iterrows():
    idx = int(row["movie_index"])
    if 0 <= idx < n_movies:
        genre_sets_arr[idx] = _parse_set_field(row.get("genres"))
        keyword_sets_arr[idx] = _parse_set_field(row.get("keywords"))
        actor_sets_arr[idx] = _parse_set_field(row.get("top_5_actors"))
        d = row.get("director", "")
        director_arr[idx] = str(d).strip().lower() if pd.notna(d) else ""
        c = row.get("collection_name", "")
        collection_arr[idx] = str(c).strip().lower() if pd.notna(c) else ""

# Load bayesian scores for Stage 1 scoring
bayesian_norm_arr = np.zeros(n_movies, dtype=np.float32)
if ITEM_FEATURES_CSV.exists():
    _if = pd.read_csv(ITEM_FEATURES_CSV)
elif ITEM_FEATURES_PARQUET.exists():
    _if = pd.read_parquet(ITEM_FEATURES_PARQUET)
else:
    _if = pd.DataFrame()

if not _if.empty and "bayesian_score_norm" in _if.columns:
    _if["movie_id"] = pd.to_numeric(_if["movie_id"], errors="coerce")
    _if = _if.dropna(subset=["movie_id"]).copy()
    for _, r in _if.iterrows():
        mid = int(r["movie_id"])
        eidx = movie_to_index.get(mid)
        if eidx is not None and 0 <= eidx < n_movies:
            bayesian_norm_arr[eidx] = float(r["bayesian_score_norm"])

# Stage 1 feature weights (from recommendation_v1.py BASE_FEATURE_WEIGHTS, excluding recency)
FEATURE_WEIGHTS = {
    "embedding": 0.33, "genre": 0.15, "keyword": 0.15,
    "actor": 0.10, "director": 0.05, "collection": 0.03, "bayesian": 0.15,
}
WEIGHT_SUM = sum(FEATURE_WEIGHTS.values())

print(f"Embeddings loaded: {embeddings_f32.shape} ({embeddings_f32.dtype})")
print(f"Movies with embedding index: {len(movie_to_index):,}")
print(f"Metadata arrays populated for {sum(1 for s in genre_sets_arr if s):,} movies")

# ============================================================
# Build Stage 1 candidate cache: for each unique query movie,
# compute Stage 1 top-K candidates (same as inference pipeline)
# ============================================================
# Collect unique query movies from positive interactions
pos_movie_ids = sorted(set(
    interactions.loc[interactions["label"] == 1, "movie_id"]
    .dropna().astype("int64").tolist()
))
# Filter to movies that have an embedding index
query_movie_ids = [mid for mid in pos_movie_ids if mid in movie_to_index]

print(f"\nBuilding Stage 1 cache for {len(query_movie_ids):,} unique query movies...")
t0 = time.time()

stage1_cache: dict[int, list[int]] = {}  # query_movie_id -> list of candidate movie_ids (ordered)

for batch_start in range(0, len(query_movie_ids), STAGE1_BATCH_SIZE):
    batch_mids = query_movie_ids[batch_start : batch_start + STAGE1_BATCH_SIZE]
    batch_emb_idx = np.array([movie_to_index[mid] for mid in batch_mids], dtype=np.int64)

    # Batched dot product: (batch_size, dim) @ (dim, n_movies) -> (batch_size, n_movies)
    sims = embeddings_f32[batch_emb_idx] @ embeddings_f32.T  # shape: (B, N)

    for i, q_mid in enumerate(batch_mids):
        q_emb_idx = batch_emb_idx[i]
        sim_row = sims[i].copy()
        sim_row[q_emb_idx] = -1.0  # exclude self

        # Top retrieval pool by embedding similarity
        pool_k = min(STAGE1_RETRIEVAL_POOL, n_movies - 1)
        top_idx = np.argpartition(sim_row, -pool_k)[-pool_k:]
        top_idx = top_idx[np.argsort(sim_row[top_idx])[::-1]]

        # Compute Stage 1 multi-feature scores for these candidates
        q_genres = genre_sets_arr[q_emb_idx]
        q_keywords = keyword_sets_arr[q_emb_idx]
        q_actors = actor_sets_arr[q_emb_idx]
        q_director = director_arr[q_emb_idx]
        q_collection = collection_arr[q_emb_idx]

        scores = np.empty(len(top_idx), dtype=np.float32)
        for j, c_idx in enumerate(top_idx):
            emb_s = sim_row[c_idx]
            genre_s = _jaccard(q_genres, genre_sets_arr[c_idx])
            kw_s = _jaccard(q_keywords, keyword_sets_arr[c_idx])
            act_s = _actor_overlap(q_actors, actor_sets_arr[c_idx])
            dir_s = 1.0 if (q_director and director_arr[c_idx] == q_director) else 0.0
            col_s = 1.0 if (q_collection and collection_arr[c_idx] == q_collection) else 0.0
            bay_s = bayesian_norm_arr[c_idx]

            scores[j] = (
                FEATURE_WEIGHTS["embedding"] * emb_s
                + FEATURE_WEIGHTS["genre"] * genre_s
                + FEATURE_WEIGHTS["keyword"] * kw_s
                + FEATURE_WEIGHTS["actor"] * act_s
                + FEATURE_WEIGHTS["director"] * dir_s
                + FEATURE_WEIGHTS["collection"] * col_s
                + FEATURE_WEIGHTS["bayesian"] * bay_s
            ) / WEIGHT_SUM

        # Keep top STAGE1_CANDIDATES_PER_QUERY by multi-feature score
        keep_k = min(STAGE1_CANDIDATES_PER_QUERY, len(scores))
        best = np.argpartition(scores, -keep_k)[-keep_k:]
        best = best[np.argsort(scores[best])[::-1]]

        candidate_mids = []
        for b in best:
            c_movie_id = index_to_movie.get(int(top_idx[b]))
            if c_movie_id is not None:
                candidate_mids.append(int(c_movie_id))
        stage1_cache[q_mid] = candidate_mids

    if (batch_start // STAGE1_BATCH_SIZE) % 10 == 0:
        elapsed = time.time() - t0
        done = min(batch_start + STAGE1_BATCH_SIZE, len(query_movie_ids))
        print(f"  Stage 1 cache: {done:,}/{len(query_movie_ids):,} queries | {elapsed:.1f}s")

elapsed = time.time() - t0
print(f"Stage 1 cache built: {len(stage1_cache):,} queries in {elapsed:.1f}s")
avg_cands = np.mean([len(v) for v in stage1_cache.values()])
print(f"Avg candidates per query: {avg_cands:.1f}")

Embeddings loaded: (31167, 768) (float32)
Movies with embedding index: 30,543
Metadata arrays populated for 31,167 movies

Building Stage 1 cache for 15,319 unique query movies...
  Stage 1 cache: 256/15,319 queries | 1.3s
  Stage 1 cache: 2,816/15,319 queries | 13.2s
  Stage 1 cache: 5,376/15,319 queries | 24.8s
  Stage 1 cache: 7,936/15,319 queries | 36.6s
  Stage 1 cache: 10,496/15,319 queries | 48.1s
  Stage 1 cache: 13,056/15,319 queries | 59.5s
Stage 1 cache built: 15,319 queries in 70.0s
Avg candidates per query: 150.0


In [28]:
# ------------------------------------------------------------
# 4) Generate training pairs from Stage 1 candidate cache
#
# Streams pairs to parquet in chunks via numpy buffers to avoid
# accumulating millions of Python dicts in memory (OOM fix).
# ------------------------------------------------------------
import pyarrow as pa
import pyarrow.parquet as pq

# Build per-user rating lookup: user_id -> {movie_id: rating}
rating_by_user: dict[int, dict[int, float]] = {}
for uid, grp in interactions.groupby("user_id"):
    rating_by_user[int(uid)] = dict(zip(
        grp["movie_id"].astype("int64").tolist(),
        grp["rating"].astype("float32").tolist(),
    ))

# Collect (user, query_movie) pairs from positive interactions
pos_interactions = interactions[interactions["label"] == 1].copy()
user_pos_movies: dict[int, list[int]] = (
    pos_interactions.groupby("user_id")["movie_id"]
    .apply(lambda x: list(set(x.astype("int64").tolist())))
    .to_dict()
)

user_ids = sorted(user_pos_movies.keys())
print(f"Users with positives: {len(user_ids):,}")

# --- Chunked parquet writer ---
FLUSH_EVERY = 500_000  # flush buffer to disk every N rows

_pair_schema = pa.schema([
    ("user_id", pa.int64()),
    ("query_movie_id", pa.int64()),
    ("candidate_movie_id", pa.int64()),
    ("label", pa.int8()),
])

# Pre-allocate numpy buffers (much cheaper than list-of-dicts)
_buf_user = np.empty(FLUSH_EVERY, dtype=np.int64)
_buf_query = np.empty(FLUSH_EVERY, dtype=np.int64)
_buf_cand = np.empty(FLUSH_EVERY, dtype=np.int64)
_buf_label = np.empty(FLUSH_EVERY, dtype=np.int8)
_buf_pos = 0
_total_written = 0

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
_pq_writer = pq.ParquetWriter(OUTPUT_PATH, _pair_schema)

def _flush_buffer():
    global _buf_pos, _total_written
    if _buf_pos == 0:
        return
    table = pa.table({
        "user_id": _buf_user[:_buf_pos],
        "query_movie_id": _buf_query[:_buf_pos],
        "candidate_movie_id": _buf_cand[:_buf_pos],
        "label": _buf_label[:_buf_pos],
    }, schema=_pair_schema)
    _pq_writer.write_table(table)
    _total_written += _buf_pos
    _buf_pos = 0

def _append_record(uid, qid, cid, rel):
    global _buf_pos
    _buf_user[_buf_pos] = uid
    _buf_query[_buf_pos] = qid
    _buf_cand[_buf_pos] = cid
    _buf_label[_buf_pos] = rel
    _buf_pos += 1
    if _buf_pos >= FLUSH_EVERY:
        _flush_buffer()

skipped_no_cache = 0
skipped_no_positives = 0
skipped_no_negatives = 0
stop_early = False
progress_every = 1000
total_pairs_count = 0

for user_idx, user_id in enumerate(user_ids, start=1):
    if stop_early:
        break

    if user_idx == 1 or user_idx % progress_every == 0:
        print(
            f"Processed users: {user_idx:,}/{len(user_ids):,} | "
            f"pairs so far: {_total_written + _buf_pos:,}"
        )

    user_ratings = rating_by_user.get(user_id, {})
    user_pos = user_pos_movies.get(user_id, [])

    # Cap positives per user
    if len(user_pos) > MAX_POSITIVES_PER_USER:
        user_pos = RNG.choice(
            np.array(user_pos, dtype=np.int64), size=MAX_POSITIVES_PER_USER, replace=False
        ).tolist()

    for query_movie_id in user_pos:
        if (_total_written + _buf_pos) >= MAX_TOTAL_PAIRS:
            stop_early = True
            break

        query_movie_id = int(query_movie_id)

        # Look up Stage 1 candidates for this query movie
        candidates = stage1_cache.get(query_movie_id)
        if candidates is None:
            skipped_no_cache += 1
            continue

        # Label each candidate from user's actual ratings
        # Accumulate in a small temp list first to check min pos/neg
        group_records = []
        n_pos = 0
        n_neg = 0

        for cand_mid in candidates:
            if cand_mid == query_movie_id:
                continue  # exclude self

            user_rating = user_ratings.get(cand_mid)
            if user_rating is not None:
                relevance = rating_to_relevance(user_rating)
            else:
                relevance = 0  # unseen = not relevant

            if relevance >= 2:
                n_pos += 1
            else:
                n_neg += 1

            group_records.append((int(user_id), query_movie_id, int(cand_mid), relevance))

        # Filter: require minimum positives and negatives
        if n_pos < MIN_POSITIVES_PER_GROUP:
            skipped_no_positives += 1
            continue
        if n_neg < MIN_NEGATIVES_PER_GROUP:
            skipped_no_negatives += 1
            continue

        # Write accepted group records to buffer
        for uid_r, qid_r, cid_r, rel_r in group_records:
            _append_record(uid_r, qid_r, cid_r, rel_r)

    if stop_early:
        break

# Final flush
_flush_buffer()
_pq_writer.close()

# --- Summary stats (read back lightweight) ---
reranker_pairs = pd.read_parquet(OUTPUT_PATH)
reranker_pairs = reranker_pairs.drop_duplicates(
    subset=["user_id", "query_movie_id", "candidate_movie_id"]
).reset_index(drop=True)
reranker_pairs.to_parquet(OUTPUT_PATH, index=False)

print(f"\n{'='*60}")
print(f"Generated pairs: {len(reranker_pairs):,}")
print(f"Unique users: {reranker_pairs['user_id'].nunique():,}")
print(f"Unique queries: {reranker_pairs.groupby(['user_id','query_movie_id']).ngroups:,}")
print(f"Stopped early: {stop_early}")
print(f"Skipped (no cache): {skipped_no_cache:,}")
print(f"Skipped (no positives in Stage 1 list): {skipped_no_positives:,}")
print(f"Skipped (too few negatives): {skipped_no_negatives:,}")
print(f"\nLabel (relevance) distribution:")
print(reranker_pairs["label"].value_counts().sort_index())
grp_sizes = reranker_pairs.groupby(["user_id", "query_movie_id"]).size()
print(f"\nCandidates per query: mean={grp_sizes.mean():.1f} median={grp_sizes.median():.0f}")
grp_pos = reranker_pairs[reranker_pairs["label"] >= 2].groupby(["user_id", "query_movie_id"]).size()
print(f"Positives (rel≥2) per query: mean={grp_pos.mean():.1f}")
print(f"Saved: {OUTPUT_PATH}")
reranker_pairs.head()

Users with positives: 29,378
Processed users: 1/29,378 | pairs so far: 0
Processed users: 1,000/29,378 | pairs so far: 1,505,828
Processed users: 2,000/29,378 | pairs so far: 2,944,760
Processed users: 3,000/29,378 | pairs so far: 4,467,391
Processed users: 4,000/29,378 | pairs so far: 5,895,680
Processed users: 5,000/29,378 | pairs so far: 7,333,859
Processed users: 6,000/29,378 | pairs so far: 8,806,842
Processed users: 7,000/29,378 | pairs so far: 10,303,964
Processed users: 8,000/29,378 | pairs so far: 11,782,961
Processed users: 9,000/29,378 | pairs so far: 13,189,201
Processed users: 10,000/29,378 | pairs so far: 14,566,037

Generated pairs: 14,973,629
Unique users: 7,731
Unique queries: 100,002
Stopped early: True
Skipped (no cache): 0
Skipped (no positives in Stage 1 list): 85,756
Skipped (too few negatives): 0

Label (relevance) distribution:
label
0    14608870
1      101007
2      129699
3       47068
4       86985
Name: count, dtype: int64

Candidates per query: mean=149.7 

,user_id,query_movie_id,candidate_movie_id,label
0,27,1893,1895,3
1,27,1893,1894,0
2,27,1893,11,4
3,27,1893,1891,3
4,27,1893,1892,3


In [29]:
# 2.3 Feature vector construction (chunked pipeline)

RERANKER_FEATURES_DIR = DATA_ROOT / "processed" / "reranker_features_parts"
PAIR_BATCH_SIZE = 200_000

def iter_pair_batches(batch_size=PAIR_BATCH_SIZE):
    cols = ["user_id", "query_movie_id", "candidate_movie_id", "label"]
    if "reranker_pairs" in globals() and isinstance(reranker_pairs, pd.DataFrame):
        base = reranker_pairs[cols].copy()
        for start in range(0, len(base), batch_size):
            yield base.iloc[start : start + batch_size].copy()
        return

    try:
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(OUTPUT_PATH)
        for batch in pf.iter_batches(batch_size=batch_size, columns=cols):
            yield batch.to_pandas()
        return
    except Exception:
        pass

    base = pd.read_parquet(OUTPUT_PATH, columns=cols)
    for start in range(0, len(base), batch_size):
        yield base.iloc[start : start + batch_size].copy()

In [30]:
# 2.3.a Load static lookup tables for feature construction
from collections import OrderedDict

# --- Item features for candidate quality signals ---
if ITEM_FEATURES_CSV.exists():
    item_features = pd.read_csv(ITEM_FEATURES_CSV)
elif ITEM_FEATURES_PARQUET.exists():
    item_features = pd.read_parquet(ITEM_FEATURES_PARQUET)
else:
    raise FileNotFoundError("Missing item_features file (csv or parquet)")

item_features["movie_id"] = pd.to_numeric(item_features["movie_id"], errors="coerce")
item_features = item_features.dropna(subset=["movie_id"]).copy()
item_features["movie_id"] = item_features["movie_id"].astype("int64")
item_features = item_features.set_index("movie_id", drop=False)

candidate_avg_rating_map = item_features["avg_user_rating"].to_dict()
candidate_rating_count_map = item_features["rating_count"].to_dict()
candidate_positive_ratio_map = item_features["positive_ratio"].to_dict()
candidate_bayesian_norm_map = item_features["bayesian_score_norm"].to_dict()
candidate_popularity_rank_map = item_features["popularity_rank"].to_dict()
candidate_rating_std_map = item_features["rating_std"].to_dict() if "rating_std" in item_features.columns else {}

n_items = len(item_features)
candidate_popularity_percentile_map = {
    mid: 1.0 - (rank / n_items) for mid, rank in candidate_popularity_rank_map.items()
} if n_items > 0 else {}

# --- Movie catalog for metadata features ---
catalog = pd.read_csv(
    METADATA_CSV,
    usecols=["id", "movie_index", "title", "genres", "keywords", "top_5_actors",
             "director", "collection_name", "vote_average"],
)
catalog["id"] = pd.to_numeric(catalog["id"], errors="coerce")
catalog = catalog.dropna(subset=["id"]).copy()
catalog["id"] = catalog["id"].astype("int64")
catalog["title"] = catalog["title"].fillna("").astype(str).str.strip()
catalog["genres"] = catalog["genres"].fillna("").astype(str)
catalog["keywords"] = catalog["keywords"].fillna("").astype(str)
catalog["top_5_actors"] = catalog["top_5_actors"].fillna("").astype(str)
catalog["director"] = catalog["director"].fillna("").astype(str).str.strip().str.lower()
catalog["collection_name"] = catalog["collection_name"].fillna("").astype(str).str.strip().str.lower()
catalog["vote_average"] = pd.to_numeric(catalog["vote_average"], errors="coerce").fillna(0.0)
catalog = catalog.set_index("id", drop=False)

# Raw lookup maps
genre_map = catalog["genres"].to_dict()
keyword_map = catalog["keywords"].to_dict()
actor_map = catalog["top_5_actors"].to_dict()
director_map = catalog["director"].to_dict()
collection_map = catalog["collection_name"].to_dict()
score_map = catalog["vote_average"].to_dict()
name_map = catalog["title"].to_dict()

# Parsed lookup maps (reuse _parse_set_field from cell 6)
genre_set_map = {mid: _parse_set_field(text) for mid, text in genre_map.items()}
keyword_set_map = {mid: _parse_set_field(text) for mid, text in keyword_map.items()}
actor_set_map = {mid: _parse_set_field(text) for mid, text in actor_map.items()}

# Title tokenizer for title_token_overlap
_TOKEN_RE = re.compile(r"[a-z0-9]+")
def _tokenize_title(title: str) -> set[str]:
    return set(_TOKEN_RE.findall(title.lower()))

title_tokens_map = {mid: _tokenize_title(name) for mid, name in name_map.items()}

# Small LRU cache for per-query similarity vectors used in embedding_rank
QUERY_SIM_CACHE_MAX = 256
_query_sim_cache: OrderedDict[int, np.ndarray] = OrderedDict()

def get_query_similarities(query_movie_id: int) -> np.ndarray | None:
    query_movie_id = int(query_movie_id)
    q_emb_idx = movie_to_index.get(query_movie_id)
    if q_emb_idx is None:
        return None

    cached = _query_sim_cache.get(query_movie_id)
    if cached is not None:
        _query_sim_cache.move_to_end(query_movie_id)
        return cached

    sims_all = (embeddings_f32 @ embeddings_f32[q_emb_idx]).astype(np.float32)
    sims_all[q_emb_idx] = -1.0
    _query_sim_cache[query_movie_id] = sims_all
    if len(_query_sim_cache) > QUERY_SIM_CACHE_MAX:
        _query_sim_cache.popitem(last=False)
    return sims_all

# Active baseline weight sum (excluding recency)
BASELINE_WEIGHT_SUM = sum(v for k, v in FEATURE_WEIGHTS.items())

print(f"Static lookup tables ready  |  catalog={len(catalog):,}  embeddings={embeddings_f32.shape}")
print(f"Parsed metadata maps ready  |  genres={len(genre_set_map):,} keywords={len(keyword_set_map):,} actors={len(actor_set_map):,}")

Static lookup tables ready  |  catalog=31,167  embeddings=(31167, 768)
Parsed metadata maps ready  |  genres=30,543 keywords=30,543 actors=30,543


In [31]:
# 2.3.b Process pair batches and write feature part files
import time as _time

RERANKER_FEATURES_DIR.mkdir(parents=True, exist_ok=True)
for old in RERANKER_FEATURES_DIR.glob("part_*.parquet"):
    old.unlink()

final_cols = [
    "user_id",
    "query_movie_id",
    "candidate_movie_id",
    "label",
    # --- query-candidate similarity features ---
    "embedding_similarity",
    "genre_overlap",
    "keyword_overlap",
    "actor_overlap",
    "director_match",
    "collection_match",
    # --- candidate quality features ---
    "candidate_avg_rating",
    "candidate_rating_count",
    "candidate_positive_ratio",
    "candidate_bayesian_norm",
    "candidate_popularity_rank",
    "rating_count_log",
    "score_diff",
    # --- derived features ---
    "candidate_popularity_percentile",
    "embedding_rank",
    "similarity_percentile",
    "title_token_overlap",
    "genre_overlap_count",
    "keyword_overlap_count",
    "candidate_rating_cv",
    "baseline_score",
]

part_count = 0
total_rows = 0

for batch_idx, pairs in enumerate(iter_pair_batches(PAIR_BATCH_SIZE), start=1):
    _t0 = _time.time()
    print(f"Processing feature batch {batch_idx:,} with {len(pairs):,} rows")

    pairs["user_id"] = pd.to_numeric(pairs["user_id"], errors="coerce")
    pairs["query_movie_id"] = pd.to_numeric(pairs["query_movie_id"], errors="coerce")
    pairs["candidate_movie_id"] = pd.to_numeric(pairs["candidate_movie_id"], errors="coerce")
    pairs = pairs.dropna(subset=["user_id", "query_movie_id", "candidate_movie_id"]).copy()
    if pairs.empty:
        continue

    pairs["user_id"] = pairs["user_id"].astype("int64")
    pairs["query_movie_id"] = pairs["query_movie_id"].astype("int64")
    pairs["candidate_movie_id"] = pairs["candidate_movie_id"].astype("int64")
    pairs["label"] = pd.to_numeric(pairs["label"], errors="coerce").fillna(0).astype("int8")

    features = pairs.copy()
    features["query_index"] = features["query_movie_id"].map(movie_to_index)
    features["candidate_index"] = features["candidate_movie_id"].map(movie_to_index)

    # --- Embedding similarity (vectorized dot product) ---
    valid_idx = features["query_index"].notna() & features["candidate_index"].notna()
    features["embedding_similarity"] = 0.0
    if valid_idx.any():
        q_idx = features.loc[valid_idx, "query_index"].astype("int64").to_numpy()
        c_idx = features.loc[valid_idx, "candidate_index"].astype("int64").to_numpy()
        features.loc[valid_idx, "embedding_similarity"] = np.sum(
            embeddings_f32[q_idx] * embeddings_f32[c_idx], axis=1
        ).astype(np.float32)

    _t1 = _time.time()

    # --- Embedding rank via argsort (O(N log N) per query, NOT O(N×K) broadcast) ---
    features["embedding_rank"] = 0
    for q_mid, grp_idx in features.groupby("query_movie_id").groups.items():
        sims_all = get_query_similarities(int(q_mid))
        if sims_all is None:
            continue

        # O(N log N) argsort + O(N) rank assignment — replaces O(N×K) broadcast
        descending_order = np.argsort(-sims_all)
        all_ranks = np.empty(len(sims_all), dtype=np.int32)
        all_ranks[descending_order] = np.arange(1, len(sims_all) + 1, dtype=np.int32)

        row_idx = np.array(list(grp_idx), dtype=np.int64)
        cand_emb_idx = pd.to_numeric(
            features.loc[row_idx, "candidate_index"], errors="coerce"
        ).fillna(-1).to_numpy(dtype=np.int64)
        valid = (cand_emb_idx >= 0) & (cand_emb_idx < embeddings_f32.shape[0])
        if not valid.any():
            continue

        # O(K) index lookup instead of O(N×K) comparison
        features.loc[row_idx[valid], "embedding_rank"] = all_ranks[cand_emb_idx[valid]]

    _t2 = _time.time()

    # --- Similarity percentile within query group ---
    features["similarity_percentile"] = (
        features.groupby("query_movie_id")["embedding_similarity"]
        .rank(method="average", pct=True)
        .astype("float32")
    )

    # --- Genre / keyword / actor / director / collection ---
    q_genre_sets = features["query_movie_id"].map(genre_set_map)
    c_genre_sets = features["candidate_movie_id"].map(genre_set_map)
    q_kw_sets = features["query_movie_id"].map(keyword_set_map)
    c_kw_sets = features["candidate_movie_id"].map(keyword_set_map)
    q_actor_sets = features["query_movie_id"].map(actor_set_map)
    c_actor_sets = features["candidate_movie_id"].map(actor_set_map)

    features["query_director"] = features["query_movie_id"].map(director_map)
    features["candidate_director"] = features["candidate_movie_id"].map(director_map)
    features["query_collection"] = features["query_movie_id"].map(collection_map)
    features["candidate_collection"] = features["candidate_movie_id"].map(collection_map)
    features["query_score"] = pd.to_numeric(features["query_movie_id"].map(score_map), errors="coerce")
    features["candidate_score"] = pd.to_numeric(features["candidate_movie_id"].map(score_map), errors="coerce")

    safe_q_genres = [value if isinstance(value, set) else set() for value in q_genre_sets]
    safe_c_genres = [value if isinstance(value, set) else set() for value in c_genre_sets]
    safe_q_keywords = [value if isinstance(value, set) else set() for value in q_kw_sets]
    safe_c_keywords = [value if isinstance(value, set) else set() for value in c_kw_sets]
    safe_q_actors = [value if isinstance(value, set) else set() for value in q_actor_sets]
    safe_c_actors = [value if isinstance(value, set) else set() for value in c_actor_sets]

    features["genre_overlap"] = [float(_jaccard(a, b)) for a, b in zip(safe_q_genres, safe_c_genres)]
    features["genre_overlap_count"] = [float(len(a & b)) for a, b in zip(safe_q_genres, safe_c_genres)]

    features["keyword_overlap"] = [float(_jaccard(a, b)) for a, b in zip(safe_q_keywords, safe_c_keywords)]
    features["keyword_overlap_count"] = [float(len(a & b)) for a, b in zip(safe_q_keywords, safe_c_keywords)]

    features["actor_overlap"] = [float(_actor_overlap(a, b)) for a, b in zip(safe_q_actors, safe_c_actors)]

    features["director_match"] = (
        (features["query_director"] == features["candidate_director"])
        & (features["query_director"] != "")
    ).astype("int8")

    features["collection_match"] = (
        (features["query_collection"] == features["candidate_collection"])
        & (features["query_collection"] != "")
    ).astype("int8")

    # --- Title token overlap (franchise detection) ---
    q_tokens = features["query_movie_id"].map(title_tokens_map)
    c_tokens = features["candidate_movie_id"].map(title_tokens_map)
    features["title_token_overlap"] = [
        float(_jaccard(qt if isinstance(qt, set) else set(),
                       ct if isinstance(ct, set) else set()))
        for qt, ct in zip(q_tokens, c_tokens)
    ]

    _t3 = _time.time()

    # --- Item quality features ---
    features["candidate_avg_rating"] = pd.to_numeric(
        features["candidate_movie_id"].map(candidate_avg_rating_map), errors="coerce"
    ).fillna(0.0)
    features["candidate_rating_count"] = pd.to_numeric(
        features["candidate_movie_id"].map(candidate_rating_count_map), errors="coerce"
    ).fillna(0.0)
    features["candidate_positive_ratio"] = pd.to_numeric(
        features["candidate_movie_id"].map(candidate_positive_ratio_map), errors="coerce"
    ).fillna(0.0)
    features["candidate_bayesian_norm"] = pd.to_numeric(
        features["candidate_movie_id"].map(candidate_bayesian_norm_map), errors="coerce"
    ).fillna(0.0)
    features["candidate_popularity_rank"] = pd.to_numeric(
        features["candidate_movie_id"].map(candidate_popularity_rank_map), errors="coerce"
    ).fillna(0.0)
    features["rating_count_log"] = np.log1p(features["candidate_rating_count"].astype(float))
    features["score_diff"] = (
        features["query_score"].fillna(0.0) - features["candidate_score"].fillna(0.0)
    ).abs()

    features["candidate_popularity_percentile"] = pd.to_numeric(
        features["candidate_movie_id"].map(candidate_popularity_percentile_map), errors="coerce"
    ).fillna(0.0)

    # --- candidate_rating_cv: coefficient of variation ---
    cand_std = pd.to_numeric(
        features["candidate_movie_id"].map(candidate_rating_std_map), errors="coerce"
    ).fillna(0.0)
    cand_avg = features["candidate_avg_rating"].copy()
    features["candidate_rating_cv"] = np.where(
        cand_avg > 0.01, cand_std / cand_avg, 0.0
    ).astype(np.float32)

    # --- baseline_score: Stage 1 multi-feature weighted score ---
    features["baseline_score"] = (
        FEATURE_WEIGHTS["embedding"]    * features["embedding_similarity"]
        + FEATURE_WEIGHTS["genre"]      * features["genre_overlap"]
        + FEATURE_WEIGHTS["keyword"]    * features["keyword_overlap"]
        + FEATURE_WEIGHTS["actor"]      * features["actor_overlap"]
        + FEATURE_WEIGHTS["director"]   * features["director_match"].astype("float32")
        + FEATURE_WEIGHTS["collection"] * features["collection_match"].astype("float32")
        + FEATURE_WEIGHTS["bayesian"]   * features["candidate_bayesian_norm"]
    ).astype("float32") / BASELINE_WEIGHT_SUM

    # --- Assemble output ---
    feature_df = features[final_cols].copy().fillna(0.0)
    feature_df["user_id"] = feature_df["user_id"].astype("int64")
    feature_df["query_movie_id"] = feature_df["query_movie_id"].astype("int64")
    feature_df["candidate_movie_id"] = feature_df["candidate_movie_id"].astype("int64")
    feature_df["label"] = feature_df["label"].astype("int8")
    feature_df["director_match"] = feature_df["director_match"].astype("int8")
    feature_df["collection_match"] = feature_df["collection_match"].astype("int8")

    float_cols = [
        "embedding_similarity", "genre_overlap", "keyword_overlap", "actor_overlap",
        "candidate_avg_rating", "candidate_rating_count", "candidate_positive_ratio",
        "candidate_bayesian_norm", "candidate_popularity_rank", "rating_count_log",
        "score_diff", "candidate_popularity_percentile", "similarity_percentile",
        "title_token_overlap", "genre_overlap_count", "keyword_overlap_count",
        "candidate_rating_cv", "baseline_score",
    ]
    for col in float_cols:
        feature_df[col] = pd.to_numeric(feature_df[col], errors="coerce").fillna(0.0).astype("float32")
    feature_df["embedding_rank"] = pd.to_numeric(
        feature_df["embedding_rank"], errors="coerce"
    ).fillna(0).astype("int32")

    out_path = RERANKER_FEATURES_DIR / f"part_{part_count:05d}.parquet"
    feature_df.to_parquet(out_path, index=False)

    total_rows += len(feature_df)
    part_count += 1

    _t4 = _time.time()
    print(
        f"  saved {out_path.name} | rows={len(feature_df):,} | "
        f"total={total_rows:,} | "
        f"emb={_t1-_t0:.1f}s rank={_t2-_t1:.1f}s meta={_t3-_t2:.1f}s rest={_t4-_t3:.1f}s"
    )

    del features, feature_df, pairs

print(f"\nFeature part files: {part_count}")
print(f"Total feature rows: {total_rows:,}")

Processing feature batch 1 with 200,000 rows
  saved part_00000.parquet | rows=200,000 | total=200,000 | emb=3.0s rank=9.0s meta=3.8s rest=0.7s
Processing feature batch 2 with 200,000 rows
  saved part_00001.parquet | rows=200,000 | total=400,000 | emb=2.8s rank=8.1s meta=3.5s rest=0.6s
Processing feature batch 3 with 200,000 rows
  saved part_00002.parquet | rows=200,000 | total=600,000 | emb=2.4s rank=6.7s meta=3.9s rest=0.7s
Processing feature batch 4 with 200,000 rows
  saved part_00003.parquet | rows=200,000 | total=800,000 | emb=2.9s rank=11.5s meta=3.8s rest=0.6s
Processing feature batch 5 with 200,000 rows
  saved part_00004.parquet | rows=200,000 | total=1,000,000 | emb=2.3s rank=8.0s meta=3.1s rest=0.5s
Processing feature batch 6 with 200,000 rows
  saved part_00005.parquet | rows=200,000 | total=1,200,000 | emb=2.1s rank=9.2s meta=3.3s rest=0.5s
Processing feature batch 7 with 200,000 rows
  saved part_00006.parquet | rows=200,000 | total=1,400,000 | emb=1.8s rank=5.8s meta=

In [32]:
# 2.3.c Summarize chunked output
part_files = sorted(RERANKER_FEATURES_DIR.glob("part_*.parquet"))
if not part_files:
    raise RuntimeError("No feature part files were generated.")

part_rows = []
for part in part_files:
    df_part = pd.read_parquet(part, columns=["label"])
    part_rows.append(len(df_part))

print(f"Parts generated: {len(part_files)}")
print(f"Total rows across parts: {sum(part_rows):,}")
print(f"First part: {part_files[0]}")
print(f"Last part: {part_files[-1]}")

Parts generated: 75
Total rows across parts: 14,973,629
First part: ../../data/processed/reranker_features_parts/part_00000.parquet
Last part: ../../data/processed/reranker_features_parts/part_00074.parquet


In [33]:
# 2.3.d Optional preview (first part)
preview = pd.read_parquet(part_files[0]).head()
print(f"Preview columns: {list(preview.columns)}")
preview

Preview columns: ['user_id', 'query_movie_id', 'candidate_movie_id', 'label', 'embedding_similarity', 'genre_overlap', 'keyword_overlap', 'actor_overlap', 'director_match', 'collection_match', 'candidate_avg_rating', 'candidate_rating_count', 'candidate_positive_ratio', 'candidate_bayesian_norm', 'candidate_popularity_rank', 'rating_count_log', 'score_diff', 'candidate_popularity_percentile', 'embedding_rank', 'similarity_percentile', 'title_token_overlap', 'genre_overlap_count', 'keyword_overlap_count', 'candidate_rating_cv', 'baseline_score']


,user_id,query_movie_id,candidate_movie_id,label,embedding_similarity,genre_overlap,keyword_overlap,actor_overlap,director_match,collection_match,...,rating_count_log,score_diff,candidate_popularity_percentile,embedding_rank,similarity_percentile,title_token_overlap,genre_overlap_count,keyword_overlap_count,candidate_rating_cv,baseline_score
0,27,1893,1895,3,0.699747,1.0,0.055556,0.795976,1,1,...,9.788582,0.7,0.995056,2,0.991667,0.363636,3.0,1.0,0.323946,0.681604
1,27,1893,1894,0,0.686670,1.0,0.086957,0.795976,1,1,...,9.959963,0.0,0.994724,3,0.985000,0.363636,3.0,2.0,0.368002,0.663712
2,27,1893,11,4,0.657552,1.0,0.090909,0.000000,1,1,...,11.252158,1.7,0.998854,4,0.978333,0.285714,3.0,2.0,0.237410,0.616162
3,27,1893,1891,3,0.721872,1.0,0.040000,0.000000,0,1,...,11.029601,1.8,0.998071,1,0.998333,0.100000,3.0,1.0,0.228960,0.580792
4,27,1893,1892,3,0.634468,1.0,0.055556,0.000000,0,1,...,11.046356,1.5,0.996382,5,0.971667,0.100000,3.0,1.0,0.237499,0.545166


In [34]:
# 2.3.e Note
print("Chunked features are saved as parquet parts in:")
print(RERANKER_FEATURES_DIR)
print("Use these parts directly for training, or concatenate later if needed.")

Chunked features are saved as parquet parts in:
../../data/processed/reranker_features_parts
Use these parts directly for training, or concatenate later if needed.


In [35]:
# 2.4 Query groups + 2.5 user-based train/val/test split

TRAIN_PATH = DATA_ROOT / "processed" / "reranker_train.parquet"
VAL_PATH = DATA_ROOT / "processed" / "reranker_val.parquet"
TEST_PATH = DATA_ROOT / "processed" / "reranker_test.parquet"

part_files = sorted(RERANKER_FEATURES_DIR.glob("part_*.parquet"))
if not part_files:
    raise RuntimeError("No feature part files found. Run 2.3 first.")

# 1) Collect unique users from feature parts
all_users = set()
for part in part_files:
    u = pd.read_parquet(part, columns=["user_id"])["user_id"]
    u = pd.to_numeric(u, errors="coerce").dropna().astype("int64")
    all_users.update(u.unique().tolist())

all_users = np.array(sorted(all_users), dtype=np.int64)
rng = np.random.default_rng(42)
rng.shuffle(all_users)

n_users = len(all_users)
n_train = int(n_users * 0.70)
n_val = int(n_users * 0.15)
n_test = n_users - n_train - n_val

train_users = set(all_users[:n_train].tolist())
val_users = set(all_users[n_train:n_train + n_val].tolist())
test_users = set(all_users[n_train + n_val:].tolist())

print(f"Unique users: {n_users:,}")
print(f"Train users: {len(train_users):,} | Val users: {len(val_users):,} | Test users: {len(test_users):,}")

# 2) Stream parts and write split files
train_counts = {}
val_counts = {}
test_counts = {}

def _update_counts(counter: dict, frame: pd.DataFrame) -> None:
    vc = frame.groupby(["user_id", "query_movie_id"]).size()
    for (uid, qid), cnt in vc.items():
        key = (int(uid), int(qid))
        counter[key] = counter.get(key, 0) + int(cnt)

use_pyarrow = True
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    use_pyarrow = False

for output in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if output.exists():
        output.unlink()

if use_pyarrow:
    train_writer = None
    val_writer = None
    test_writer = None
    try:
        for part in part_files:
            chunk = pd.read_parquet(part)
            chunk["user_id"] = pd.to_numeric(chunk["user_id"], errors="coerce")
            chunk = chunk.dropna(subset=["user_id"]).copy()
            chunk["user_id"] = chunk["user_id"].astype("int64")

            train_chunk = chunk[chunk["user_id"].isin(train_users)]
            val_chunk = chunk[chunk["user_id"].isin(val_users)]
            test_chunk = chunk[chunk["user_id"].isin(test_users)]

            if not train_chunk.empty:
                _update_counts(train_counts, train_chunk)
                t = pa.Table.from_pandas(train_chunk, preserve_index=False)
                if train_writer is None:
                    train_writer = pq.ParquetWriter(TRAIN_PATH, t.schema)
                train_writer.write_table(t)

            if not val_chunk.empty:
                _update_counts(val_counts, val_chunk)
                t = pa.Table.from_pandas(val_chunk, preserve_index=False)
                if val_writer is None:
                    val_writer = pq.ParquetWriter(VAL_PATH, t.schema)
                val_writer.write_table(t)

            if not test_chunk.empty:
                _update_counts(test_counts, test_chunk)
                t = pa.Table.from_pandas(test_chunk, preserve_index=False)
                if test_writer is None:
                    test_writer = pq.ParquetWriter(TEST_PATH, t.schema)
                test_writer.write_table(t)
    finally:
        if train_writer is not None:
            train_writer.close()
        if val_writer is not None:
            val_writer.close()
        if test_writer is not None:
            test_writer.close()
else:
    train_chunks = []
    val_chunks = []
    test_chunks = []
    for part in part_files:
        chunk = pd.read_parquet(part)
        chunk["user_id"] = pd.to_numeric(chunk["user_id"], errors="coerce")
        chunk = chunk.dropna(subset=["user_id"]).copy()
        chunk["user_id"] = chunk["user_id"].astype("int64")

        train_chunk = chunk[chunk["user_id"].isin(train_users)]
        val_chunk = chunk[chunk["user_id"].isin(val_users)]
        test_chunk = chunk[chunk["user_id"].isin(test_users)]

        if not train_chunk.empty:
            _update_counts(train_counts, train_chunk)
            train_chunks.append(train_chunk)
        if not val_chunk.empty:
            _update_counts(val_counts, val_chunk)
            val_chunks.append(val_chunk)
        if not test_chunk.empty:
            _update_counts(test_counts, test_chunk)
            test_chunks.append(test_chunk)

    if train_chunks:
        pd.concat(train_chunks, ignore_index=True).to_parquet(TRAIN_PATH, index=False)
    if val_chunks:
        pd.concat(val_chunks, ignore_index=True).to_parquet(VAL_PATH, index=False)
    if test_chunks:
        pd.concat(test_chunks, ignore_index=True).to_parquet(TEST_PATH, index=False)

# 3) LambdaRank query groups
train_group_sizes = np.array([train_counts[uid] for uid in sorted(train_counts)], dtype=np.int64)
val_group_sizes = np.array([val_counts[uid] for uid in sorted(val_counts)], dtype=np.int64)
test_group_sizes = np.array([test_counts[uid] for uid in sorted(test_counts)], dtype=np.int64)

print(f"Saved: {TRAIN_PATH}")
print(f"Saved: {VAL_PATH}")
print(f"Saved: {TEST_PATH}")
print(f"Train groups: {len(train_group_sizes):,} | example: {train_group_sizes[:10]}")
print(f"Val groups: {len(val_group_sizes):,} | example: {val_group_sizes[:10]}")
print(f"Test groups: {len(test_group_sizes):,} | example: {test_group_sizes[:10]}")

def _safe_count_rows(path: Path) -> int:
    if not path.exists():
        return 0
    return len(pd.read_parquet(path, columns=["user_id"]))

print(f"Train rows: {_safe_count_rows(TRAIN_PATH):,}")
print(f"Val rows: {_safe_count_rows(VAL_PATH):,}")
print(f"Test rows: {_safe_count_rows(TEST_PATH):,}")

Unique users: 7,731
Train users: 5,411 | Val users: 1,159 | Test users: 1,161
Saved: ../../data/processed/reranker_train.parquet
Saved: ../../data/processed/reranker_val.parquet
Saved: ../../data/processed/reranker_test.parquet
Train groups: 70,008 | example: [150 149 150 150 150 150 149 150 150 150]
Val groups: 14,567 | example: [150 150 150 150 150 150 150 149 150 150]
Test groups: 15,427 | example: [150 150 150 150 150 150 150 150 148 150]
Train rows: 10,482,567
Val rows: 2,181,105
Test rows: 2,309,957


### Reranker Training

In [ ]:
# 4.1 Train LambdaRank model (graded relevance)
from pathlib import Path
import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
except Exception as exc:
    raise ImportError(
        "lightgbm is required. Install it in this notebook environment (e.g., pip install lightgbm)."
    ) from exc

TRAIN_PATH = DATA_ROOT / "processed" / "reranker_train.parquet"
VAL_PATH = DATA_ROOT / "processed" / "reranker_val.parquet"
TEST_PATH = DATA_ROOT / "processed" / "reranker_test.parquet"

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing split file: {p}")

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

required_cols = {"user_id", "label"}
for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = required_cols - set(frame.columns)
    if missing:
        raise ValueError(f"{name} split missing required columns: {missing}")

# Sort by (user_id, query_movie_id) so each query forms one contiguous LambdaRank group
train_df = train_df.sort_values(["user_id", "query_movie_id"]).reset_index(drop=True)
val_df = val_df.sort_values(["user_id", "query_movie_id"]).reset_index(drop=True)
test_df = test_df.sort_values(["user_id", "query_movie_id"]).reset_index(drop=True)

group_train = train_df.groupby(["user_id", "query_movie_id"]).size().to_numpy(dtype=np.int64)
group_val = val_df.groupby(["user_id", "query_movie_id"]).size().to_numpy(dtype=np.int64)
group_test = test_df.groupby(["user_id", "query_movie_id"]).size().to_numpy(dtype=np.int64)

# Exclude ID columns and label
exclude_cols = {
    "label", "user_id", "movie_id",
    "query_movie_id", "candidate_movie_id",
}
candidate_feature_cols = [c for c in train_df.columns if c not in exclude_cols]

numeric_feature_cols = [c for c in candidate_feature_cols if pd.api.types.is_numeric_dtype(train_df[c])]
dropped_non_numeric = [c for c in candidate_feature_cols if c not in numeric_feature_cols]

feature_cols = numeric_feature_cols
if not feature_cols:
    raise ValueError("No numeric feature columns found after exclusions.")

X_train = train_df[feature_cols].fillna(0.0).to_numpy(dtype=np.float32)
y_train = train_df["label"].to_numpy(dtype=np.int32)
X_val = val_df[feature_cols].fillna(0.0).to_numpy(dtype=np.float32)
y_val = val_df["label"].to_numpy(dtype=np.int32)
X_test = test_df[feature_cols].fillna(0.0).to_numpy(dtype=np.float32)
y_test = test_df["label"].to_numpy(dtype=np.int32)

# Verify graded labels are present
print(f"Label value counts (train): {np.unique(y_train, return_counts=True)}")

train_data = lgb.Dataset(
    X_train,
    label=y_train,
    group=group_train,
    feature_name=feature_cols,
    free_raw_data=False,
)

val_data = lgb.Dataset(
    X_val,
    label=y_val,
    group=group_val,
    reference=train_data,
    feature_name=feature_cols,
    free_raw_data=False,
)

params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5, 10, 20],
    "label_gain": [0, 1, 3, 7, 15],  # exponential gains for relevance 0-4
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[val_data],
    valid_names=["val"],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)

print(f"\nTrain rows: {len(train_df):,} | Val rows: {len(val_df):,} | Test rows: {len(test_df):,}")
print(f"Train groups: {len(group_train):,} | Val groups: {len(group_val):,} | Test groups: {len(group_test):,}")
print(f"Features used ({len(feature_cols)}): {feature_cols}")
if dropped_non_numeric:
    print(f"Dropped non-numeric features ({len(dropped_non_numeric)}): {dropped_non_numeric}")

In [15]:
# 4.2 Evaluation metrics: NDCG@K, MAP@10, HitRate@10 (graded relevance aware)

def _group_index_ranges(group_sizes: np.ndarray):
    start = 0
    for g in group_sizes:
        end = start + int(g)
        yield start, end
        start = end

def _dcg_at_k(labels_sorted: np.ndarray, k: int, label_gain: list[int] | None = None) -> float:
    """DCG with optional custom label gains for graded relevance."""
    labels = labels_sorted[:k]
    if labels.size == 0:
        return 0.0
    if label_gain is not None:
        gains = np.array([label_gain[min(int(l), len(label_gain)-1)] for l in labels], dtype=np.float64)
    else:
        gains = (2 ** labels - 1).astype(np.float64)
    discounts = np.log2(np.arange(2, labels.size + 2))
    return float(np.sum(gains / discounts))

# Use same label_gain as training for consistent evaluation
EVAL_LABEL_GAIN = [0, 1, 3, 7, 15]

def _ndcg_at_k(y_true: np.ndarray, y_pred: np.ndarray, group_sizes: np.ndarray, k: int) -> float:
    scores = []
    for s, e in _group_index_ranges(group_sizes):
        labels = y_true[s:e]
        preds = y_pred[s:e]
        if labels.size == 0:
            continue
        order_pred = np.argsort(-preds)
        order_ideal = np.argsort(-labels)
        dcg = _dcg_at_k(labels[order_pred], k, EVAL_LABEL_GAIN)
        idcg = _dcg_at_k(labels[order_ideal], k, EVAL_LABEL_GAIN)
        scores.append(0.0 if idcg == 0 else dcg / idcg)
    return float(np.mean(scores)) if scores else 0.0

def _map_at_k(y_true: np.ndarray, y_pred: np.ndarray, group_sizes: np.ndarray, k: int, threshold: int = 2) -> float:
    """MAP@K treating relevance >= threshold as positive."""
    ap_scores = []
    for s, e in _group_index_ranges(group_sizes):
        labels = y_true[s:e]
        preds = y_pred[s:e]
        if labels.size == 0:
            continue
        order = np.argsort(-preds)[:k]
        ranked = labels[order]
        pos = 0
        precision_sum = 0.0
        for i, rel in enumerate(ranked, start=1):
            if rel >= threshold:
                pos += 1
                precision_sum += pos / i
        ap_scores.append(0.0 if pos == 0 else precision_sum / pos)
    return float(np.mean(ap_scores)) if ap_scores else 0.0

def _hit_rate_at_k(y_true: np.ndarray, y_pred: np.ndarray, group_sizes: np.ndarray, k: int, threshold: int = 2) -> float:
    """HitRate@K treating relevance >= threshold as a hit."""
    hits = []
    for s, e in _group_index_ranges(group_sizes):
        labels = y_true[s:e]
        preds = y_pred[s:e]
        if labels.size == 0:
            continue
        order = np.argsort(-preds)[:k]
        hits.append(1.0 if np.any(labels[order] >= threshold) else 0.0)
    return float(np.mean(hits)) if hits else 0.0

def evaluate_ranking(y_true: np.ndarray, y_pred: np.ndarray, group_sizes: np.ndarray) -> dict:
    return {
        "NDCG@5": _ndcg_at_k(y_true, y_pred, group_sizes, 5),
        "NDCG@10": _ndcg_at_k(y_true, y_pred, group_sizes, 10),
        "NDCG@20": _ndcg_at_k(y_true, y_pred, group_sizes, 20),
        "MAP@10": _map_at_k(y_true, y_pred, group_sizes, 10),
        "HitRate@10": _hit_rate_at_k(y_true, y_pred, group_sizes, 10),
    }

val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

val_metrics = evaluate_ranking(y_val, val_pred, group_val)
test_metrics = evaluate_ranking(y_test, test_pred, group_test)

print("Validation metrics:")
for k, v in val_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

metric_targets = {
    "NDCG@10": 0.70,
    "NDCG@20": 0.65,
    "MAP@10": 0.50,
    "HitRate@10": 0.80,
}

print("\nAgainst targets (test):")
for metric, target in metric_targets.items():
    value = test_metrics[metric]
    status = "PASS" if value >= target else "MISS"
    print(f"  {metric}: {value:.4f} (target>{target:.2f}) [{status}]")

Validation metrics:
  NDCG@5: 0.5582
  NDCG@10: 0.6123
  NDCG@20: 0.6516
  MAP@10: 0.5925
  HitRate@10: 0.9129

Test metrics:
  NDCG@5: 0.5591
  NDCG@10: 0.6143
  NDCG@20: 0.6530
  MAP@10: 0.5915
  HitRate@10: 0.9173

Against targets (test):
  NDCG@10: 0.6143 (target>0.70) [MISS]
  NDCG@20: 0.6530 (target>0.65) [PASS]
  MAP@10: 0.5915 (target>0.50) [PASS]
  HitRate@10: 0.9173 (target>0.80) [PASS]


In [17]:
# 4.2.b Sanity checks — graded relevance distribution
def _group_relevance_stats(df: pd.DataFrame) -> dict:
    grp = df.groupby(["user_id", "query_movie_id"])
    sizes = grp.size()
    pos_counts = grp["label"].apply(lambda x: (x >= 2).sum())
    return {
        "candidates_per_group_mean": sizes.mean(),
        "candidates_per_group_median": sizes.median(),
        "positives_per_group_mean": pos_counts.mean(),
        "pct_groups_with_pos": (pos_counts > 0).mean() * 100,
    }

for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    self_mask = (df["query_movie_id"] == df["candidate_movie_id"]) if {"query_movie_id", "candidate_movie_id"}.issubset(df.columns) else pd.Series(False, index=df.index)
    stats = _group_relevance_stats(df)
    print(f"[{split_name}] rows={len(df):,} users={df['user_id'].nunique():,}")
    print(f"  self-candidate rows: {int(self_mask.sum()):,} ({self_mask.mean()*100:.2f}%)")
    print(f"  candidates/group: mean={stats['candidates_per_group_mean']:.1f} median={stats['candidates_per_group_median']:.0f}")
    print(f"  positives(rel≥2)/group: mean={stats['positives_per_group_mean']:.2f}")
    print(f"  groups with ≥1 positive: {stats['pct_groups_with_pos']:.1f}%")
    print(f"  relevance distribution:")
    vc = df["label"].value_counts().sort_index()
    for rel_val, count in vc.items():
        print(f"    rel={rel_val}: {count:,} ({count/len(df)*100:.1f}%)")
    print()

# Feature means by relevance level — verify similarity features are now properly correlated
print("=== Feature means by relevance (train) ===")
rel_levels = sorted(train_df["label"].unique())
check_features = ["embedding_similarity", "genre_overlap", "baseline_score",
                   "candidate_bayesian_norm", "candidate_popularity_percentile"]
for feat in check_features:
    if feat not in train_df.columns:
        continue
    vals = [f"{train_df.loc[train_df['label']==r, feat].mean():.4f}" for r in rel_levels]
    print(f"  {feat:40s} " + " | ".join(f"rel={r}:{v}" for r, v in zip(rel_levels, vals)))

[train] rows=10,484,012 users=5,456
  self-candidate rows: 0 (0.00%)
  candidates/group: mean=149.7 median=150
  positives(rel≥2)/group: mean=2.62
  groups with ≥1 positive: 100.0%
  relevance distribution:
    rel=0: 10,230,862 (97.6%)
    rel=1: 69,420 (0.7%)
    rel=2: 90,001 (0.9%)
    rel=3: 32,228 (0.3%)
    rel=4: 61,501 (0.6%)

[val] rows=2,228,831 users=1,169
  self-candidate rows: 0 (0.00%)
  candidates/group: mean=149.7 median=150
  positives(rel≥2)/group: mean=2.65
  groups with ≥1 positive: 100.0%
  relevance distribution:
    rel=0: 2,175,333 (97.6%)
    rel=1: 14,115 (0.6%)
    rel=2: 19,218 (0.9%)
    rel=3: 6,745 (0.3%)
    rel=4: 13,420 (0.6%)

[test] rows=2,260,507 users=1,170
  self-candidate rows: 0 (0.00%)
  candidates/group: mean=149.7 median=150
  positives(rel≥2)/group: mean=2.63
  groups with ≥1 positive: 100.0%
  relevance distribution:
    rel=0: 2,205,518 (97.6%)
    rel=1: 15,310 (0.7%)
    rel=2: 19,592 (0.9%)
    rel=3: 6,900 (0.3%)
    rel=4: 13,187 (0.

In [ ]:
# 4.2.c Verification: feature correlations with relevance
# With Stage 1 simulation, similarity features should now be POSITIVELY correlated
# with relevance (higher similarity = more likely to be relevant)
from scipy import stats as spstats

print("Spearman correlation of features with relevance (test set):")
for feat in feature_cols:
    rho, pval = spstats.spearmanr(test_df[feat].fillna(0), test_df["label"])
    marker = "***" if abs(rho) > 0.05 else ""
    print(f"  {feat:40s} ρ={rho:+.4f}  p={pval:.2e} {marker}")

In [20]:
# 4.3 Baseline comparison — Stage 1 baseline_score as the baseline ranker
# This is the most meaningful comparison: can the reranker beat Stage 1's own scoring?

baseline_col = "baseline_score" if "baseline_score" in test_df.columns else None

if baseline_col is None:
    print("No baseline_score column found in test split. Skipping baseline comparison.")
    print("  available columns:", sorted(test_df.columns.tolist()))
else:
    baseline_metrics = evaluate_ranking(
        y_test, test_df[baseline_col].to_numpy(dtype=np.float32), group_test
    )
    print(f"Baseline column: '{baseline_col}' (Stage 1 multi-feature weighted score)")
    print("\nBaseline vs Reranker (test):")
    print(f"  {'Metric':15s} {'Baseline':>10s} {'Reranker':>10s} {'Relative':>10s}")
    print(f"  {'─'*15} {'─'*10} {'─'*10} {'─'*10}")
    for metric in ["NDCG@5", "NDCG@10", "NDCG@20", "MAP@10", "HitRate@10"]:
        b = baseline_metrics.get(metric, 0.0)
        r = test_metrics.get(metric, 0.0)
        rel = ((r - b) / b * 100.0) if b > 0 else np.nan
        rel_str = f"{rel:+.2f}%" if np.isfinite(rel) else "n/a"
        print(f"  {metric:15s} {b:10.4f} {r:10.4f} {rel_str:>10s}")

    ndcg10_rel = (
        (test_metrics["NDCG@10"] - baseline_metrics["NDCG@10"])
        / baseline_metrics["NDCG@10"] * 100.0
    ) if baseline_metrics.get("NDCG@10", 0) > 0 else np.nan
    if np.isfinite(ndcg10_rel):
        print(f"\nNDCG@10 relative lift: {ndcg10_rel:+.2f}%")
        print("Deployment criterion (>= +5%):", "PASS" if ndcg10_rel >= 5.0 else "MISS")

Baseline column: 'baseline_score' (Stage 1 multi-feature weighted score)

Baseline vs Reranker (test):
  Metric            Baseline   Reranker   Relative
  ─────────────── ────────── ────────── ──────────
  NDCG@5              0.1953     0.5591   +186.28%
  NDCG@10             0.2210     0.6143   +177.98%
  NDCG@20             0.2552     0.6530   +155.87%
  MAP@10              0.2642     0.5915   +123.87%
  HitRate@10          0.4788     0.9173    +91.58%

NDCG@10 relative lift: +177.98%
Deployment criterion (>= +5%): PASS


In [ ]:
# 4.5 Optional hyperparameter tuning scaffold (manual grid)
search_space = {
    "num_leaves": [15, 31, 63],
    "learning_rate": [0.01, 0.05, 0.1],
    "min_data_in_leaf": [10, 20, 50],
}

print("Manual tuning space:")
for k, v in search_space.items():
    print(f"  {k}: {v}")
print("Tip: keep early stopping enabled; compare by validation NDCG@10.")

In [18]:
# 4.4 Feature importance analysis
importance_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance_gain": model.feature_importance(importance_type="gain"),
        "importance_split": model.feature_importance(importance_type="split"),
    }
).sort_values("importance_gain", ascending=False).reset_index(drop=True)

print("Top 20 features by gain:")
display(importance_df.head(20))

popularity_like = importance_df[importance_df["feature"].str.contains("popularity|rating_count|bayesian|score", case=False, na=False)]
print(f"Popularity-like features in top 20: {sum(popularity_like.index < 20)}")

Top 20 features by gain:


,feature,importance_gain,importance_split
0,candidate_rating_count,3.552294e+06,2910
1,rating_count_log,7.788300e+05,585
2,candidate_popularity_rank,2.315036e+05,1665
3,candidate_positive_ratio,1.139531e+05,1891
4,baseline_score,9.973767e+04,3450
5,candidate_popularity_percentile,7.989569e+04,1517
6,genre_overlap_count,7.708752e+04,502
7,score_diff,7.538362e+04,2482
8,embedding_similarity,5.897156e+04,3548
9,candidate_avg_rating,5.009970e+04,1952


Popularity-like features in top 20: 7


In [19]:
# 4.6 Save model + metadata
RERANKER_DIR = ARTIFACT_DIR / "reranker_v2"
RERANKER_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = RERANKER_DIR / "model.txt"
FEATURES_PATH = RERANKER_DIR / "feature_columns.json"
IMPORTANCE_PATH = RERANKER_DIR / "feature_importance.csv"

model.save_model(str(MODEL_PATH))
importance_df.to_csv(IMPORTANCE_PATH, index=False)

import json
with open(FEATURES_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "feature_columns": feature_cols,
        "label_gain": [0, 1, 3, 7, 15],
        "relevance_mapping": {
            "5.0": 4, "4.5": 3, "4.0": 2, "3.0-3.5": 1, "<3.0 or unseen": 0
        },
        "stage1_candidates_per_query": STAGE1_CANDIDATES_PER_QUERY,
    }, f, ensure_ascii=False, indent=2)

print(f"Saved model: {MODEL_PATH}")
print(f"Saved feature columns: {FEATURES_PATH}")
print(f"Saved feature importance: {IMPORTANCE_PATH}")

Saved model: ../../ml/artifacts/content_recommendation_v2/reranker_v2/model.txt
Saved feature columns: ../../ml/artifacts/content_recommendation_v2/reranker_v2/feature_columns.json
Saved feature importance: ../../ml/artifacts/content_recommendation_v2/reranker_v2/feature_importance.csv


## Phase 4 — Model Training & Evaluation (v2: Stage 1 simulation)

### Key change from v1
Training pairs now come from **simulated Stage 1 candidate lists** — candidates are the same embedding-similar movies that the inference pipeline retrieves. Labels use **graded relevance** (0-4) from actual user ratings.

### 4.1 LightGBM LambdaRank
Train a ranking model with query groups defined by `(user_id, query_movie_id)`. Uses custom `label_gain = [0, 1, 3, 7, 15]` for graded relevance.

### 4.2 Evaluation metrics
Compute `NDCG@5/10/20`, `MAP@10`, and `HitRate@10` on validation/test splits. DCG uses the same label gains as training.

### 4.3 Baseline comparison
Compare reranker metrics with Stage 1 `baseline_score` column — this is the most meaningful comparison since the reranker's job is to improve on Stage 1 ordering.

### 4.4 Feature importance
Inspect model feature importance and verify similarity features now contribute positively.

### 4.6 Save model
Persist model artifact to `ml/artifacts/content_recommendation_v2/reranker_v2/model.txt`.